# Simple viz with ephys for scrolling through time

In [3]:
import fastplotlib as fpl 
import numpy as np
import zmq
import tifffile
import scipy
import queue 
from ipywidgets import IntSlider, VBox, Layout
from scipy.ndimage import gaussian_filter1d
from real_spike.utils import *
import h5py
from PIL import Image

In [4]:
f = h5py.File("/home/clewis/wasabi/reaganbullins2/ProjectionProject/rb50/20250125/MAT_FILES/rb50_20250125_datastruct_pt3.mat", 'r')
data = f['data']
print(data.keys())

<KeysViewHDF5 ['aligned_cue_rec_time', 'aligned_laser_rec_time', 'any', 'chan_spk', 'cue', 'cue_rec_time', 'cue_trial_time', 'depth_spk', 'field_id', 'field_size', 'full_field', 'grab', 'grab_ms', 'laser', 'laser_rec_time', 'laser_trial_time', 'lift', 'lift_ms', 'lift_rec_time', 'mouth', 'mouth_ms', 'no_success', 'pattern_fill', 'pattern_id', 'pattern_xy', 'plift', 'plift_ms', 'single', 'spikes_raw_cue', 'spikes_raw_cue_extended', 'spikes_raw_laser', 'spikes_raw_lift', 'totTime', 'trial_start']>


# Get a trial of data

In [5]:
file_path = Path("/home/clewis/wasabi/reaganbullins2/ProjectionProject/rb50/20250125/rb50_20250125_g0/rb50_20250125_g0_t0.imec0.ap.bin")
meta_path = Path("/home/clewis/wasabi/reaganbullins2/ProjectionProject/rb50/20250125/rb50_20250125_g0/rb50_20250125_g0_t0.imec0.ap.meta")

In [6]:
meta_data = get_meta(meta_path)

In [7]:
ap_data = get_sample_data(file_path, meta_data)
ap_data.shape

(385, 164723792)

## Get conversion params

In [8]:
vmax = float(meta_data["imAiRangeMax"])
# get Imax
imax = float(meta_data["imMaxInt"])
# get gain
gain = float(meta_data['imroTbl'].split(sep=')')[1].split(sep=' ')[3])

(vmax, imax, gain)

(0.6, 512.0, 500.0)

In [9]:
j = 18
c = data["aligned_cue_rec_time"][j, 0] 
g = data["grab_ms"][j, 0]
# 50 ms before cue
cue_time = int((c - 50) / 1_000 * 30_000)

# end at 5m after grab 
end_behavior = int((c + g + 200) / 1_000 * 30_000)


trial = ap_data[50:200, cue_time:end_behavior]

# convert to microvolts
conv_data = 1e6 * trial / vmax / imax / gain
# high pass filter 
filt_data = butter_filter(conv_data, 1_000, 30_000)

# get 1 second before the median 
m_start = cue_time - (30 * 1000)
# convert to microvolts, high pass filter
trial_median = ap_data[50:200, m_start:cue_time]
trial_median = 1e6 * trial_median / vmax / imax / gain
trial_median = butter_filter(trial_median, 1_000, 30_000)

# calculate the median
median = np.median(trial_median, axis=1)

# # get spike times
# spike_ixs, counts = get_spike_events(filt_data, median)

# a = np.zeros((filt_data.shape[0], filt_data.shape[1]))

# for i, sc in enumerate(spike_ixs):
#     a[i, sc] = 1

# b = 1 * 30 # 30ms per bin
# binned_spikes = bin_spikes(a, b)

In [10]:
COLORS = np.random.rand(150, 4) # [n_colors, rgba] array
COLORS[:, -1] = 1 # set alpha = 1

# Create figure

In [11]:
filt_data.shape

(150, 19800)

In [12]:
window_size = 20 * 30

In [13]:
figure = fpl.Figure(size=(1000, 900), 
                    names=["filtered spikes", "raster", "smoothed spikes"],
                    extents=[(0, 0.5, 0, 0.7), (0.5, 1, 0, 0.7), (0, 1, .7, 1)]
               )


init_data = filt_data[:, :window_size]
lg = figure["filtered spikes"].add_line_stack(init_data, colors="gray", thickness=2, separation=30)

ixs, _ = get_spike_events(init_data, median)

for i in range(len(ixs)):
        if ixs[i].shape[0] == 0:
            continue
        lg[i].colors[ixs[i]] = "orange"

spikes, colors = make_raster(ixs, COLORS)
spikes = np.concatenate(spikes)

figure["raster"].add_scatter(spikes, sizes=3, colors=colors)

a = np.zeros((init_data.shape[0], init_data.shape[1]))

for i, sc in enumerate(ixs):
    a[i, sc] = 1

b = 2 # 30ms per bin
binned_spikes = bin_spikes(a, b)

smooth_spikes = gaussian_filter1d(a, axis=1, sigma=5)
figure["smoothed spikes"].add_line_collection(smooth_spikes, thickness=2, colors=COLORS)


for s in figure:
    s.axes.visible = False
    s.camera.maintain_aspect = False

RFBOutputContext()

Detected skylake derivative running on mesa i915. Clears to srgb textures will use manual shader clears.
/home/clewis/repos/fastplotlib/fastplotlib/graphics/features/_base.py:18: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")
/home/clewis/repos/fastplotlib/fastplotlib/graphics/features/_base.py:18: UserWarning: casting int64 array to float32
  warn(f"casting {array.dtype} array to float32")


In [16]:
slider = IntSlider(value=0, min=0, max=int(filt_data.shape[1] / window_size) - 1, layout=Layout(width="55%"))

In [17]:
def update(ev):
    global filt_data, lg, median, COLORS
    t = ev["new"] 

    init_data = filt_data[:, int(t*window_size):int((t+1)* window_size)] 
    for i in range(lg.data[:].shape[0]):
        # lg[i].data[:-30, 1] = lg[i].data[30:, 1]
        # lg[i].data[-30:, 1] = chunk[i]
        lg[i].data[:, 1] = init_data[i]

    lg.colors = "gray"


    ixs, _ = get_spike_events(init_data, median) 
    
    for i in range(len(ixs)):
        if ixs[i].shape[0] == 0:
            continue
        lg[i].colors[ixs[i]] = "orange"

    # make a raster plot using the pre-defined channel colors
    spikes, colors = make_raster(ixs, COLORS)
    spikes = np.concatenate(spikes)

    # clear the old raster plot
    figure["raster"].clear()
    figure["smoothed spikes"].clear()

    # add new raster
    figure["raster"].add_scatter(spikes, sizes=3, colors=colors)

    a = np.zeros((init_data.shape[0], init_data.shape[1]))

    for i, sc in enumerate(ixs):
        a[i, sc] = 1
    
    b = 2 # 30ms per bin
    binned_spikes = bin_spikes(a, b)
    
    smooth_spikes = gaussian_filter1d(a, axis=1, sigma=5)
    figure["smoothed spikes"].add_line_collection(smooth_spikes, thickness=2, colors=COLORS)
    

In [18]:
slider.observe(update, "value")

In [19]:
VBox([figure.show(), slider])

In [23]:
figure.export("/home/clewis/Desktop/prelim_figures/fpl_B.jpg")